# Lab 09: How to answer follow-up questions with retrieval

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | In class**

## Goal

Carry a user's meaning from earlier turns into a search query, then inspect whether the retrieved evidence supports
an answer. After a Boreal password-reset conversation, *How long is the link valid?* retrieves Atlas's policy first.
The answer gives Atlas's 30 minutes with a real quote. Rewriting the question to name Boreal produces the relevant
10-minute answer in the saved examples.

Search receives a string, not the conversation in your head. We will first see what the last turn loses, then make
the missing product and task explicit. The self-study parts let a model propose its own search through a tool,
compare those queries on seven conversations, and ask for clarification when the product remains uncertain.

## What you will learn

- Trace how a follow-up can lose its product or task while a quoted citation still passes its check.
- Rewrite a conversation's last turn as a standalone query while retaining its account type and conditions.
- Check product names and added digits, and explain why these checks cannot establish meaning or factual support.
- Inspect a public LangChain tool's schema, bind it to a model, and execute a proposed call with its matching result.
- Compare routes on the same conversations and distinguish repeated calls from additional test cases.
- Ask which product the user means, and separate ambiguity from a gap in the available evidence.

**In class:** Parts A to C, except the ten-run count at the end of Part A. That subsection and Parts D, E and F are
self-study.


## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-27
OPENAI_API_KEY    found in .env


Config identifies the chat and embedding models and their endpoints. `JSON_MODE` selects how the cited answer
is structured. A **follow-up** depends on earlier turns: *What about Boreal?* supplies a product but omits the task.
Retrieval across conversation turns is called **conversational retrieval** or **multi-turn retrieval**.

The notebook embeds 277 documents once, roughly 21,000 tokens, and embeds a query on each search. The full path
makes roughly 120 chat calls if each tool interaction needs one search and one final reply; completing the challenges
adds calls. This is an estimate of work, not a measured bill. The tool loop can request more searches.

To estimate cost, add input and output tokens over **every** chat call, including rewriting and tool proposals,
then add corpus and query embedding tokens. For rates quoted per million tokens:

$$
\text{cost} = \frac{T_{\text{chat in}}p_{\text{in}} + T_{\text{chat out}}p_{\text{out}}
+ T_{\text{embed}}p_{\text{embed}}}{10^6}
$$

Here $T$ counts tokens and $p$ is the applicable price per million. Chat input includes instructions, history,
tool schemas and returned passages where supplied; later tool calls resend the accumulated messages. These cells
do not total usage or cached-token discounts, so no precise dollar total follows from the call estimate. Check the
rates for your configured destination before running. Repeating a cell repeats its calls.

The saved outputs use the models shown in Config. Compare the retrieved order, query meaning and reply with the
explanations; wording and similarity scores can vary.


---

## Part A: Trace what a follow-up loses when searched alone

A user can know which link they mean while the search receives too few words to identify it. We will follow one
conversation through retrieval, the actual evidence lines, a cited answer and a quote check before counting repeats.

`resources/conversations.json` contains seven **constructed** help-desk exchanges about Atlas and Boreal, rather
than recorded customer chats. Earlier assistant messages are supplied text, not replies generated during this run.
Load one complete conversation first:


In [3]:
# conversations: the seven conversations in the file, a list of dictionaries; by_id finds one by its id
import json

LAB = ROOT / "09_Follow_Up_Questions"
conversations = json.loads((LAB / "resources" / "conversations.json").read_text(encoding="utf-8"))["conversations"]
by_id = {conversation["id"]: conversation for conversation in conversations}

# case: the conversation Parts A to C follow
case = by_id["link_after_boreal"]
case

{'id': 'link_after_boreal',
 'slice': 'reference resolution',
 'turns': [{'role': 'user', 'content': 'How do I reset my Boreal password?'},
  {'role': 'assistant',
   'content': 'Boreal password reset uses a mobile app [H04].'},
  {'role': 'user', 'content': 'How long is the link valid?'}],
 'products': ['Boreal'],
 'source_id': 'H04'}

The displayed dictionary keeps the conversation and its evaluation labels together:

| Field | Type and role |
|---|---|
| `id` | A string naming the case, used to join queries and checks later |
| `slice` | A string naming the behaviour under test; *reference resolution* means identifying what a later phrase refers to |
| `turns` | A list in chronological order; each dictionary holds a `role` and a string `content` |
| `products` | A list of required product names for checking the query, here `['Boreal']` |
| `source_id` | The answering policy's ID, here `'H04'`, or `None` when no single answering policy is labelled |

`case['turns'][-1]` selects the last message; negative indexing counts back from the end of a list. Here it is
*How long is the link valid?* The preceding user question names Boreal. `products` and `source_id` are an answer
key used after generation; the model receives turns or a query, never these labels.

The supplied assistant turn cites H04. That citation helps describe what was discussed; it does not turn the earlier
reply into fresh policy evidence. We still need to retrieve and read the policy. List the last turn of all seven cases:


In [4]:
# One line per conversation: id, slice, products, the answering policy, the last turn
for conversation in conversations:
    print(f"{conversation['id']:18}{conversation['slice']:22}{conversation['products']!s:20}"
          f"{conversation['source_id']!s:6}{conversation['turns'][-1]['content']}")

topic_switch      topic switch          ['Boreal']          H04   What about Boreal?
link_after_boreal reference resolution  ['Boreal']          H04   How long is the link valid?
account_type      reference resolution  ['Atlas']           H01   Can temporary accounts do that?
pronoun           reference resolution  ['Atlas']           H03   Who can export all of them?
earlier_answer    reference resolution  ['Atlas']           H05   Where can I download them?
evidence_gap      evidence gap          ['Boreal']          None  What about Boreal?
two_products      ambiguous             ['Atlas', 'Boreal'] None  How long is the link valid?


Five cases have an answering policy. `evidence_gap` asks about Boreal billing, which no policy covers.
`two_products` is labelled ambiguous because either person's reset link could be meant; its `products` list requires
both names for the query check, but a query containing both still leaves the user intent unresolved.

Reuse the LangChain pipeline demonstrated in Lab 05 and Lab 06 Part D:

| Operation | Input → output |
|---|---|
| `load_corpus()` then `to_documents()` | 272 chunk records and five policies → 277 `Document` objects with IDs and text |
| `build_store()` | Documents → an `InMemoryVectorStore`, using `embed_documents()` as demonstrated in Lab 05 |
| `retrieve()` | Store and one query string → three `(Document, cosine score)` pairs, best first |
| `evidence()` | Those ordered pairs → a list of full `[ID] text` strings, with no token budget |
| `answer()` | Question string and evidence strings → one structured `CitedAnswer` |

Lab 05 used `OpenAIEmbeddings` directly before `lc_embeddings()` and demonstrated the store's public search method.
These helpers reuse those operations and settings. The new operation we are learning here is resolving the follow-up,
so first build the same store:


In [5]:
# store: the 277 documents (272 chunks and the five policies), embedded into a fresh InMemoryVectorStore
from shared import lc_pipeline

store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
print(len(store.store), "documents in the store")

277 documents in the store


The store contains 277 documents. This count checks how many entries were indexed; it does not verify their
contents or ID/text alignment. Search with the last turn exactly as typed:

> How long is the link valid?

Which policy will rank first, Atlas's H02 or Boreal's H04? `show()` below prints the score, ID and first 70 characters
of each candidate. That preview identifies a hit; the next step displays the complete evidence before answering.


In [6]:
# show() prints one line per hit; each hit is a (Document, cosine score) pair
def show(hits):
    for document, score in hits:
        print(f"  {score:.4f}  {document.id:13}{document.page_content[:70]}")

# follow_up: the last turn as typed; hits: the three closest documents for it
follow_up = case["turns"][-1]["content"]
hits = lc_pipeline.retrieve(store, follow_up)
show(hits)

  0.5949  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3770  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.1892  H01          Atlas passwords can be reset by email. Temporary accounts require a su


H02, Atlas's link policy, ranks first at about 0.59; H04, Boreal's, ranks second at about 0.38.
The last turn contains neither product name. The ranking shows how this embedding search matched that wording;
it does not reveal the model's internal reason or imply a probability that H02 is correct.

Each hit pairs a LangChain `Document` with a cosine score. `document.id` identifies the passage and
`document.page_content` holds its text. Turn the three hits into the exact evidence strings the answering step uses:


In [7]:
# lines: the three hits as "[ID] text" lines, the evidence the answer step reads
lines = lc_pipeline.evidence(hits)
for line in lines:
    print(line)

[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[H04] Boreal reset links expire after 10 minutes. Password reset uses a mobile app.
[H01] Atlas passwords can be reset by email. Temporary accounts require a support agent.


Read the two durations beside their product names: H02 says Atlas, 30 minutes; H04 says Boreal, 10 minutes.
H01 adds Atlas's temporary-account exception. `evidence()` preserves this order and includes full text, rather than
the shortened preview printed by `show()`.

The next call receives `follow_up`, *How long is the link valid?*, and these three strings. `answer()` formats its
user message as `Question: ...` followed by `Evidence:` and the lines, alongside the shared system rules. It receives
no earlier turns and no `products` answer key. Inspect one reply before introducing the repeat helper:


In [8]:
# reply: one cited answer for the follow-up as typed
reply = lc_pipeline.answer(follow_up, lines, settings)
reply

CitedAnswer(answer='Atlas reset links expire after 30 minutes.', claims=[Claim(claim='Atlas reset links expire after 30 minutes.', source_id='H02', quote='Atlas reset links expire after 30 minutes.')])

The returned `CitedAnswer` has an `.answer` string and a `.claims` list. Each `Claim` has three string fields:
`.claim` is the assertion, `.source_id` names a line, and `.quote` contains the words offered as support. The saved
reply says *Atlas reset links expire after 30 minutes.* and cites H02.

Lab 02's `quote_found()` checks whether a claim's quote occurs in its named evidence line after lower-casing and
word normalization. It also accepts cuts marked with `...` when the remaining parts occur in order. It does not
compare the claim's meaning with the quote or resolve which product the user meant.

`claim.model_dump()` converts the Pydantic `Claim` into the dictionary this checker expects. Check the one claim:


In [9]:
# For each claim: the cited ID, whether its quote is in that evidence line, and the quote
from shared.answer import quote_found

for claim in reply.claims:
    print(claim.source_id, quote_found(claim.model_dump(), lines), "|", claim.quote)

H02 True | Atlas reset links expire after 30 minutes.


`H02 True` means the quoted Atlas sentence occurs in the H02 line. This quote is authentic within the supplied
evidence, but it answers the wrong conversation: the user meant Boreal. A product-aware reading of the history
reveals what the occurrence check cannot.

Now reuse the demonstrated answer call and quote check in `answers()`. It repeats the same question and evidence,
then groups identical **answer-and-check strings** with `Counter`. Different claim lists or wording can form separate
keys even when their broad meaning agrees. It prints occurrences, rather than a correctness score:


In [10]:
# answers() asks the same question `runs` times and counts each different answer, with its citations checked
from collections import Counter

def answers(question, hits, runs=3):
    lines = lc_pipeline.evidence(hits)
    seen = Counter()
    for _ in range(runs):
        reply = lc_pipeline.answer(question, lines, settings)
        checked = [(claim.source_id, quote_found(claim.model_dump(), lines)) for claim in reply.claims]
        seen[f"{reply.answer} | cited, quote found: {checked}"] += 1
    for text, count in seen.items():
        print(f"  {count} of {runs}: {text}")

### Count repeated replies to one fixed input

`evidence(hits)` is computed once inside `answers()`. Each loop iteration makes a fresh answer call on those same
lines; `checked` records each cited ID and its quote-occurrence result. Adding 1 to `seen[key]` counts an occurrence,
and `seen.items()` returns each distinct key with its count.

No reply is appended to the conversation. Three calls here are three repeats of one input, not three independent
conversations or three successive user turns. Try the follow-up three times:


In [11]:
# Three answers to the follow-up as typed, from the same three hits
answers(follow_up, hits)

  2 of 3: Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes. | cited, quote found: [('H02', True), ('H04', True)]
  1 of 3: Atlas reset links expire after 30 minutes. | cited, quote found: [('H02', True)]


The saved repeats give Atlas alone 1 of 3 and both products 2 of 3. Every printed quote check is True;
none of the replies selects Boreal alone. Giving both correctly assigned durations leaves the user to identify the
applicable one; it differs from asserting a false duration, but still fails to resolve this conversation's intent.

Compare a second case before summarising. In `topic_switch`, the three turns are:

> user: How long does an Atlas reset link remain valid?  
> assistant: Atlas reset links expire after 30 minutes [H02].  
> user: What about Boreal?

The last turn supplies the new product but leaves the reset-link task in the earlier question. Will naming Boreal
alone be enough for retrieval and answering?


In [12]:
# switch: the topic_switch conversation; its last turn searched as typed, then answered three times
switch = by_id["topic_switch"]
switch_hits = lc_pipeline.retrieve(store, switch["turns"][-1]["content"])
show(switch_hits)
answers(switch["turns"][-1]["content"], switch_hits)

  0.3256  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.3202  S16-P01-C01  rainforest spanned a relatively narrow band. It expanded again during 
  0.2958  S16-P01-C00  Following the Cretaceous–Paleogene extinction event, the extinction of


  3 of 3: Insufficient evidence. | cited, quote found: []


H04 ranks just above an unrelated rainforest chunk, yet the answer refuses all three calls. Naming Boreal got
the right policy first; the remaining question, *What about Boreal?*, still omits what to ask about it. Retrieval
success and answer success therefore differ even in this small example.

For `link_after_boreal`, the last turn loses the product; for `topic_switch`, it loses the task. The next part
restores that missing context before both search and answering. The quote check cannot repair either omission.

### Checkpoint A

| You should see | Our run |
|---|---|
| Conversations | 7, from `topic_switch` to `two_products` |
| *How long is the link valid?* as typed | H02 0.5949, H04 0.3770, H01 0.1892 |
| One answer, quote check | *Atlas reset links expire after 30 minutes.*, cites H02, quote found `True` |
| Three answers | Atlas's 30 minutes alone 1 of 3; both products' times 2 of 3; every quote found |
| *What about Boreal?* as typed | H04 0.3256, then S16-P01-C01 0.3202 (*rainforest spanned ...*); *Insufficient evidence.* 3 of 3 |

### Check repeatability of the as-typed answer (self-study)

The next cell adds ten answer calls on the original query and fixed hits. It can reveal variation missed by the
three-call example. It adds no new conversation, wording or evidence, so its denominator is ten calls on one case.


In [13]:
# Ten answers to the follow-up as typed, to see the spread three runs can hide
answers(follow_up, hits, runs=10)

  2 of 10: Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes. | cited, quote found: [('H02', True), ('H04', True)]
  8 of 10: Atlas reset links expire after 30 minutes. | cited, quote found: [('H02', True)]


The saved ten repeats give Atlas alone 8/10 and both correctly assigned times 2/10. Neither outcome
selects Boreal alone. Read these as occurrences on the same fixed evidence, rather than a population error rate.
Even ten matching replies would not test another wording or another conversation.

| You should see | Our run |
|---|---|
| Ten answers as typed | Atlas's 30 minutes alone 8 of 10; both products' times 2 of 10 |

---

## Part B: Carry the product and task into a standalone query

The two failures suggest what search needs: a product **and** the task inherited from earlier turns.
**Query rewriting** adds one chat call to express the follow-up as a **standalone query**, understandable without
history. For our first case, *Boreal password reset link validity duration* retains Boreal and link duration while
leaving the actual duration to the evidence.

The conversation is already visible in Part A. Convert its ordered turn dictionaries into a readable string,
one `role: content` line per turn. The generator inside `join()` creates each line; `"\n"` separates the lines:


In [14]:
# history_text() writes the turns as one string, one "role: content" line per turn
def history_text(turns):
    return "\n".join(f"{turn['role']}: {turn['content']}" for turn in turns)

print(history_text(case["turns"]))

user: How do I reset my Boreal password?
assistant: Boreal password reset uses a mobile app [H04].
user: How long is the link valid?


The three lines preserve who said what and their order. They remain text to be rewritten. The prefixes `user:`
and `assistant:` inside that string are not separate API message roles.

A good rewrite keeps the **latest intent**, rather than copying everything. In this case it keeps Boreal and the
password-reset link task; it can drop the earlier app statement and citation because neither is the requested
duration. For `topic_switch`, it must keep the duration task but replace Atlas with Boreal. Copying the earlier
30-minute answer would import the wrong product's fact.

The instruction also names account types and conditions. *Can temporary accounts do that?* must become a question
about **temporary Atlas accounts resetting by email**, not Atlas resets generally. H01's exception changes the
answer to requiring a support agent. Build the messages before invoking the model:


In [15]:
# REWRITE: the system instruction; messages: the instruction, then the whole conversation as one user message
REWRITE = ("Rewrite the last user turn as a standalone search query. "
           "Keep products, account types and conditions. "
           "Do not answer it. Return only the query.")
messages = [("system", REWRITE), ("user", history_text(case["turns"]))]
messages

[('system',
  'Rewrite the last user turn as a standalone search query. Keep products, account types and conditions. Do not answer it. Return only the query.'),
 ('user',
  'user: How do I reset my Boreal password?\nassistant: Boreal password reset uses a mobile app [H04].\nuser: How long is the link valid?')]

`messages` is a list of two `(role, text)` tuples, the short form accepted by LangChain's chat interface.
The system message gives the rewrite task; the user message contains the whole conversation as quoted input.
No retrieved policy text or answer-key fields are included in this rewrite request.

Lab 05 demonstrated `ChatOpenAI.invoke()` directly and then centralised its configured constructor in
`lc_chat_model()`. We reuse that object here. Its public `invoke(messages)` returns an `AIMessage`; `.content`
is the query text for these calls. There is no answer schema on this rewrite. Send the inspected messages once:


In [16]:
# chat: the chat model from models.env; query: the rewritten question, the text of the reply
from shared.settings import lc_chat_model

chat = lc_chat_model("CHAT_MODEL", settings)
query = chat.invoke(messages).content
print(query)

Boreal password reset link validity duration.


The saved query is *Boreal password reset link validity duration.* A search query can be a phrase; its job is to
preserve the question's meaning, rather than grammatical form. It adds no proposed answer.

Feed that string to the same retriever. The cell prints candidates first, then passes the **rewritten query** and
their full evidence lines to `answers()`. The answering step still sees no conversation history:


In [17]:
# query_hits: the three closest documents for the rewritten query; then three answers to it
query_hits = lc_pipeline.retrieve(store, query)
show(query_hits)
answers(query, query_hits)

  0.7825  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.5801  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.5124  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: Boreal reset links expire after 10 minutes. | cited, quote found: [('H04', True)]


H04 now ranks first at about 0.78 and H02 second at about 0.58. All three saved answers give Boreal's 10 minutes
with an H04 quote that occurs in the evidence. The query and candidate texts explain this improvement on this case;
three equal answers do not establish a general success rate.

The two-message rewrite request is now demonstrated. `rewrite(turns)` packages exactly that call for reuse.
Apply it to `topic_switch`, where the missing information was the task rather than the product:


In [18]:
# rewrite() makes Part B's call for any conversation's turns and returns the query
def rewrite(turns):
    return chat.invoke([("system", REWRITE), ("user", history_text(turns))]).content

# switch_query: the rewrite of topic_switch
switch_query = rewrite(switch["turns"])
print(switch_query)

How long does a Boreal reset link remain valid?


The new query, *How long does a Boreal reset link remain valid?*, combines the duration task from the first user
turn with Boreal from the last. It drops Atlas and the assistant's 30 minutes. Search and answer using this query:


In [19]:
# The rewritten topic_switch query: its three closest documents, then three answers
switch_hits = lc_pipeline.retrieve(store, switch_query)
show(switch_hits)
answers(switch_query, switch_hits)

  0.6734  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.6418  H02          Atlas reset links expire after 30 minutes. Request a new link if the o
  0.3650  H01          Atlas passwords can be reset by email. Temporary accounts require a su


  3 of 3: Boreal reset links remain valid for 10 minutes. | cited, quote found: [('H04', True)]


H04 leads at about 0.67, close to H02 at about 0.64, and all three saved answers give Boreal's 10 minutes.
The inherited task and new product are explicit in the answer input now. A high similarity score alone did not
establish that result: we also read the retrieved policy and the actual reply.

Both Part B examples work here. A rewrite can still omit a condition or copy an earlier answer, so we next inspect
its text with small checks rather than treating a successful example as a guarantee.

### Checkpoint B

| You should see | Our run |
|---|---|
| Rewrite of *How long is the link valid?* | *Boreal password reset link validity duration.* |
| Its search and answers | H04 0.7825, H02 0.5801; *Boreal reset links expire after 10 minutes.* 3 of 3, quote found |
| Rewrite of *What about Boreal?* | *How long does a Boreal reset link remain valid?*; H04 0.6734, H02 0.6418; Boreal's 10 minutes 3 of 3 |

---

## Part C: Screen rewritten queries for missing products and added digits

A rewrite is model output. We can detect some mistakes before using it, but retaining words does not establish
that it retained the user's meaning. We will write two small, deliberately limited checks and try passing and failing
inputs before applying them to more conversations.

- `products_kept()` tests for each product named by the case's answer key in the query, ignoring case.
- `new_numbers()` lists digit sequences in the query absent from all user turns. It can catch an earlier assistant's
  duration or citation ID being copied, but it is not a general test that a query contains no answer.

Start with Part B's query and the known Boreal label:


In [20]:
# products_kept() is True when every product in the answer key appears in the query, upper or lower case
def products_kept(query, case):
    return all(product.lower() in query.lower() for product in case["products"])

print(f"{query!r}: product kept {products_kept(query, case)}")

'Boreal password reset link validity duration.': product kept True


`all()` combines one boolean per required product; `.lower()` makes the comparison case-insensitive.
This is substring matching: it neither recognises aliases nor rejects an additional wrong product. For the
ambiguous case, naming both products passes, even though intent remains unresolved.

Next, inspect added digits. `re.findall(r"\d+", text)` returns a list of digit strings: `\d` means a digit and
`+` means one or more in sequence. Thus `H05` contributes `'05'`, while `monthly` contributes nothing:


In [21]:
# new_numbers() lists the numbers in the query that none of the user's turns contains
import re

def new_numbers(query, turns):
    typed = re.findall(r"\d+", " ".join(turn["content"] for turn in turns if turn["role"] == "user"))
    return [number for number in re.findall(r"\d+", query) if number not in typed]

print(f"{query!r}: new numbers {new_numbers(query, case['turns'])}")

'Boreal password reset link validity duration.': new numbers []


`typed` collects digits from **user** messages only; assistant messages are excluded so their numbers are not
automatically accepted. The returned list keeps each query digit string absent from `typed`. Part B's query gives
`[]`, an empty list, because it contains no digits.

This is a screen for added digit strings, not provenance verification. A user-typed number could be copied into a
wrong relationship and pass. A supported number expressed as a word could be flagged if rewritten as digits.
Test two deliberately bad queries against the appropriate conversation:


In [22]:
# Two bad queries written by hand, each checked against its own conversation
for bad, bad_case in (("How long is the link valid?", case),
                      ("Do Boreal reset links also expire after 30 minutes?", switch)):
    print(f"{bad!r}: product kept {products_kept(bad, bad_case)} | new numbers {new_numbers(bad, bad_case['turns'])}")

'How long is the link valid?': product kept False | new numbers []
'Do Boreal reset links also expire after 30 minutes?': product kept True | new numbers ['30']


The as-typed question fails the product test and adds no digits. The second query contains Boreal but imports
Atlas's 30-minute duration; it passes the product test and returns `['30']` from the digit test.

Now consider a **hypothetical** rewrite of `account_type`: *Can Atlas accounts reset by email?* It names Atlas and
adds no digits, so it would pass both tests, yet it loses *temporary*. H01's actual text says:

> Atlas passwords can be reset by email. Temporary accounts require a support agent.

The omitted condition changes which sentence answers. No such omission is demonstrated in the saved rewrites;
this example explains a check's blind spot. Read the original turns beside the query to judge product, task, account
type and conditions. These two functions alone cannot certify a rewrite or the truth of a later answer.

The checks are diagnostic here: a failed result is printed, not used to block a search. Part E applies them to
84 query instances; Part F's final function also does not enforce them. A system that rejects or clarifies on a
failed screen would need that control flow explicitly.

### Checkpoint C

| Input | Product kept | Added digits |
|---|---|---|
| Part B's Boreal rewrite | `True` | `[]` |
| As-typed link question | `False` | `[]` |
| Boreal question with Atlas's duration | `True` | `['30']` |


---

## Part D: Inspect and execute a model-proposed search (self-study)

A second route gives the chat model a search tool so it can propose the query while reading the conversation.
Retrieval used this way is a component of **agentic RAG**. We will inspect the public tool interface and perform one
round trip manually before introducing a loop.

A tool is a callable function with a description and argument schema. LangChain's
[`@tool`](https://docs.langchain.com/oss/python/langchain/tools) decorator wraps our function. `query: str`
defines one string input; `-> str` describes the result. The function reuses the retrieval and evidence operations
already demonstrated: search three candidates, format them with IDs, then join the evidence list into one string.

`convert_to_openai_tool()` prints the schema sent alongside the messages. Defining the tool does not execute a search:


In [23]:
# search_documents: Lab 05's search as a LangChain tool; its docstring is the description the model reads
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def search_documents(query: str) -> str:
    """Search the help-centre documents and return the three closest passages. The query must make sense on its own."""
    return "\n".join(lc_pipeline.evidence(lc_pipeline.retrieve(store, query)))

# What the model is sent for this tool
print(convert_to_openai_tool(search_documents))

{'type': 'function', 'function': {'name': 'search_documents', 'description': 'Search the help-centre documents and return the three closest passages. The query must make sense on its own.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}


The printed dictionary describes the interface the model can request:

| Field | Meaning |
|---|---|
| `type: 'function'` | A function tool supplied by our application |
| `function.name` | `search_documents`, the name a proposed call must use |
| `description` | The docstring explaining when and how to search |
| `parameters` | An object schema with a required string field `query` |

The model receives this description and schema, not the Python function body or vector store.
[`bind_tools()`](https://docs.langchain.com/oss/python/langchain/models#tool-calling) returns a runnable chat
model configuration that includes the tool interface on invocation. Binding alone sends no request and runs no tool.

This route supplies the actual conversation messages after `SYSTEM`. `*case['turns']` unpacks the list's entries in
order, rather than nesting a list inside the message list. Compare this with Part B, which packed all turns into
one text string. Invoke the bound model once and inspect its proposed arguments:


In [24]:
# SYSTEM: the rules for the tool route; with_search: the chat model with the tool attached
SYSTEM = ("You answer questions about the Atlas and Boreal products. Search the documents before you answer."
          " Answer only from the passages, citing their IDs.")
with_search = chat.bind_tools([search_documents])

# tool_messages: the system message, then the three turns; reply: the model's first reply
tool_messages = [{"role": "system", "content": SYSTEM}, *case["turns"]]
reply = with_search.invoke(tool_messages)
print("text:", repr(reply.content))
print("tool calls:", reply.tool_calls)

text: ''
tool calls: [{'name': 'search_documents', 'args': {'query': 'link validity'}, 'id': 'call_UDCNswiERmp67OgV5n9o7fMZ', 'type': 'tool_call'}]


The saved `AIMessage` has empty text and one item in `.tool_calls`. That dictionary includes `name`, `args`,
`id` and `type: 'tool_call'`. `args` is a dictionary containing the model-written query; `id` identifies this request
so its result can be matched later. Empty text here accompanies a proposed operation, not a completed answer.

`reply.tool_calls[0]` selects the first call and assumes one exists. Read the printed list first; a model can instead
reply without calling a tool. Extract this call's query and apply the same diagnostic checks as in Part C:


In [25]:
# tool_query: the query in the model's tool call, checked like a rewrite
tool_query = reply.tool_calls[0]["args"]["query"]
print(f"{tool_query!r}: product kept {products_kept(tool_query, case)} | new numbers {new_numbers(tool_query, case['turns'])}")

'link validity': product kept False | new numbers []


The query is *link validity*: `product kept False`, with no added digits. The proposal has lost Boreal even
though the model read its name in history. No search has run at this point, and printing the failed check has not
blocked execution.

We deliberately execute it to observe the consequence. The public `search_documents.invoke()` receives the complete
tool-call dictionary, passes its `args` to the wrapped function and returns a `ToolMessage`. The message carries the
retrieved text and the original call ID. Inspect both before returning the result to the model:


In [26]:
# tool_message: the search result for the model's call, as a ToolMessage with the call's ID
tool_message = search_documents.invoke(reply.tool_calls[0])
print(type(tool_message).__name__, "| same ID as the call:", tool_message.tool_call_id == reply.tool_calls[0]["id"])
print(tool_message.content)

ToolMessage | same ID as the call: True
[H02] Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
[S19-P02-C01] leaky bucket. In case of a shared physical medium (such as radio or 10BASE5), the packets may be delivered according to a multiple access scheme.
[S19-P00-C01] the development of telecommunications in the Bell System. The new concept found little resonance among network implementers until the independent work of Donald Davies at the National Physical Laboratory (United Kingdom) (NPL) in the late 1960s. Davies is credited with coining the modern name packet switching and inspiring numerous packet switching networks in Europe in the decade following, including the incorporation of the concept in the early ARPANET in the United States.


The returned candidates are H02 and two packet-switching chunks. Their full text is visible above; H04 is absent.
The matching call ID proves the result is associated with the proposal. It does not verify relevance, authenticity
of the source corpus, or factual support for an answer.

`tool_messages + [reply, tool_message]` creates the next input: the system instruction and original conversation,
then the assistant's proposed call and its result. Unlike Part B's answering step, this model still sees the earlier
assistant replies. Ask for the next response, then inspect whether it proposes another call or supplies text:


In [27]:
# final: the model's next reply, after reading its own tool call and the search result
final = with_search.invoke(tool_messages + [reply, tool_message])
print("tool calls:", final.tool_calls)
print(final.content)

tool calls: []
The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].


The saved response has no further tool calls and gives Atlas's 30 minutes with H02. The citation points to the
retrieved Atlas policy; the full Boreal conversation did not prevent the wrong-product answer.

A next response can propose another search. We have now used `@tool`, `bind_tools()` and both kinds of `invoke()`
directly on visible inputs and outputs. `tool_answer()` combines this round trip: append proposals and their results,
repeat while calls remain, and return `(queries, final_text)`. The first item is a list of query strings in execution
order; the second is the final message text.


In [28]:
# tool_answer() runs the tool route to the end and returns (the queries the model wrote, its final answer)
def tool_answer(turns, model):
    messages = [{"role": "system", "content": SYSTEM}, *turns]
    reply = model.invoke(messages)
    queries = []
    while reply.tool_calls:
        messages.append(reply)
        for call in reply.tool_calls:
            queries.append(call["args"]["query"])
            messages.append(search_documents.invoke(call))
        reply = model.invoke(messages)
    return queries, reply.content

### Follow the messages through repeated tool calls

1. The first `model.invoke()` reads `SYSTEM` and the supplied turns. `queries` starts empty.
2. While the response contains calls, append that assistant response to `messages`.
3. For each call, retain its query and append the `ToolMessage` returned by `search_documents.invoke(call)`.
4. Invoke the model on the accumulated messages again. Every call must have a matching result before this request.
5. A response without tool calls ends the loop, even if its text is wrong or empty.

This demonstration has one tool and assumes the proposed name and arguments fit it. It has no loop limit or
factual-answer check; an application would need a stopping policy before letting the loop run unattended.
Each `tool_answer()` invocation starts from the supplied turns again. Repeat this fixed conversation three times:


In [29]:
# The tool route, three times on link_after_boreal
for _ in range(3):
    print(tool_answer(case["turns"], with_search))

(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


(['link validity'], 'The reset links for Atlas expire after 30 minutes. If the link has expired, you will need to request a new one [H02].')


Each saved repeat searches *link validity* and answers for Atlas. Part B's `REWRITE` instruction does not apply
to this route: here the query instructions are in the tool description and system message.

Change only the description to request the product and task. `search_documents.func` is the original Python
function. `tool('search_documents', description=PRODUCT_RULE)(...)` wraps that same function with the same schema
and name, allowing the demonstrated loop to execute it unchanged. Bind the revised tool description before
repeating the conversation:


In [30]:
# search_named: the same search with a description that names the product; with_named_search: the model with it
PRODUCT_RULE = ("Search the help-centre documents and return the three closest passages. Write a query that makes"
                " sense on its own: name the product (Atlas or Boreal) and the task.")
search_named = tool("search_documents", description=PRODUCT_RULE)(search_documents.func)
with_named_search = chat.bind_tools([search_named])

# The tool route with the new description, three times
for _ in range(3):
    print(tool_answer(case["turns"], with_named_search))

(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


(['Boreal password reset link validity'], 'The Boreal password reset links expire after 10 minutes [H04].')


The revised description produces *Boreal password reset link validity* and Boreal's 10-minute answer in all three
saved repeats. The search function and corpus have not changed. This is an observed improvement on one fixed
conversation, not a guarantee that a description enforces product preservation.

The rewrite route makes a dedicated query call before search; its answer step sees only query and evidence.
The tool route proposes a query while reading history, executes it in our code, and reads history plus results on
the next call. Those are different prompts and answer formats, so a difference in answers cannot be attributed to
query wording alone. The tool route's free text is not a `CitedAnswer`, and this loop does not run `quote_found()`
on it. Read the evidence and product relationship separately.

### Checkpoint D

| Stage | Saved observation |
|---|---|
| Tool interface | `search_documents`, a description, one required `query` string |
| First proposal | `link validity`, no text, product check `False` |
| Manual execution | Matching `ToolMessage` ID; H02 and two unrelated chunks, no H04 |
| Initial-description loop | Atlas's 30 minutes, 3 of 3 |
| Product-description loop | Boreal's 10 minutes with H04, 3 of 3 |


---

## Part E: Compare query failures on the same seven conversations (self-study)

One successful rewrite can hide another conversation's lost condition or copied answer. We will retain each query
before reducing checks to counts, and compare routes within the same case.

The four routes are **as typed**, **rewrite**, the original **tool call**, and the tool call with the **product rule**.
For the tool routes this comparison reads only the **first proposed query**. `tool_query()` invokes the bound model,
but never executes the search tool or generates a final answer. With no call it returns `''`; with multiple calls it
reads only the first. This is narrower than Part D's complete loop:


In [31]:
# tool_query(): the query in the model's first tool call, or "" when it called no tool
def tool_query(turns, model):
    calls = model.invoke([{"role": "system", "content": SYSTEM}, *turns]).tool_calls
    return calls[0]["args"]["query"] if calls else ""

print(repr(tool_query(case["turns"], with_search)))

'link validity'


The returned string is *link validity*, the first proposal seen in Part D. No tool search ran.

Each route must accept a list of turn dictionaries and return one query string. `lambda turns: ...` defines a short
function in place. For example, `lambda turns: turns[-1]['content']` returns the last turn unchanged. The dictionary
below associates each route's name with its function; the functions are called later, not during definition:


In [32]:
# routes: the four ways to write a query, each a function of the turns
routes = {"as typed": lambda turns: turns[-1]["content"],
          "rewrite": rewrite,
          "tool call": lambda turns: tool_query(turns, with_search),
          "product rule": lambda turns: tool_query(turns, with_named_search)}
list(routes)

['as typed', 'rewrite', 'tool call', 'product rule']

For one case, apply each route three times to its same original turns. Then repeat across all seven cases.
The as-typed route simply returns the same string, with no model call. The other three routes make a chat call per
repeat. Keep a `Counter` for each `(conversation_id, route)` pair so we can inspect the actual queries before checks:


In [33]:
# queries: for each (conversation id, route), a Counter of the three queries that route wrote
queries = {}
for conversation in conversations:
    for route, write in routes.items():
        queries[(conversation["id"], route)] = Counter(write(conversation["turns"]) for _ in range(3))

# One line per conversation and route
for (case_id, route), counts in queries.items():
    print(f"{case_id:18}{route:14}" + " | ".join(f"{n}x {q!r}" for q, n in counts.items()))

topic_switch      as typed      3x 'What about Boreal?'
topic_switch      rewrite       3x 'How long does a Boreal reset link remain valid?'
topic_switch      tool call     3x 'Boreal reset link validity'
topic_switch      product rule  3x 'Boreal reset link validity'
link_after_boreal as typed      3x 'How long is the link valid?'
link_after_boreal rewrite       3x 'Boreal password reset link validity duration.'
link_after_boreal tool call     3x 'link validity'
link_after_boreal product rule  3x 'Boreal password reset link validity'
account_type      as typed      3x 'Can temporary accounts do that?'
account_type      rewrite       3x 'Can temporary accounts reset Atlas passwords by email?'
account_type      tool call     3x 'temporary accounts reset password email'
account_type      product rule  3x 'temporary accounts reset Atlas password by email'
pronoun           as typed      3x 'Who can export all of them?'
pronoun           rewrite       3x 'Who can export all reports in Atla

### Read the per-conversation queries before scoring them

`queries[(case_id, route)]` is a `Counter`: query string → frequency. Seven cases and four routes give 28 counters;
three query instances per counter give 84 instances, of which 63 require model calls. Equal strings are grouped,
not discarded. A `3x` label means the same query occurred in all three calls for that case and route.

For `account_type`, the rewrite says *Can temporary accounts reset Atlas passwords by email?* The product rule's
query also retains *temporary*, Atlas and email. The original tool query retains temporary/email but drops Atlas.
None of these outputs demonstrates the planned loss of *temporary accounts*. Read that condition directly because
the two automated checks do not test it.

For `link_after_boreal`, compare the rows as a pair: the original tool writes *link validity* and the revised tool
writes *Boreal password reset link validity*. That connects the route change to the failure already inspected in D.
Now inspect `earlier_answer`, where a historical answer can enter the query instead:


In this copy, two of the three `earlier_answer` rewrites include *[H05]* from the earlier assistant turn.
The user asked where to download invoices; *monthly* and the citation were not needed to preserve that request.

The next table checks each query counter:

| Column | What is counted | What remains unchecked |
|---|---|---|
| `product kept` | Instances containing every required product name | Task, account type, extra wrong product, resolved intent |
| `no new number` | Instances whose added-digit list is empty | Copied words such as *monthly*, or misuse of an existing number |
| `right policy first` | Instances whose nearest document ID equals the labelled `source_id` | Full candidate relevance, answer correctness and factual support |

`first_id()` searches with `k=1`. The count is shown as `-` for `evidence_gap` and `two_products`, which have no
single answering-policy label. They remain in product/digit counts. The code still performs their searches; a dash
is an unavailable target comparison, not a successful refusal or clarification.


In [34]:
# first_id(): the ID of the closest document for a query (None for an empty query)
from shared.settings import chat_url

def first_id(query):
    return lc_pipeline.retrieve(store, query, k=1)[0][0].id if query else None

# One row per conversation and route: of the three queries, how many pass each test
print(f"{'conversation':18}{'route':14}{'product kept':>13}{'no new number':>15}{'right policy first':>20}")
for (case_id, route), counts in queries.items():
    conversation = by_id[case_id]
    kept = sum(n for q, n in counts.items() if products_kept(q, conversation))
    clean = sum(n for q, n in counts.items() if not new_numbers(q, conversation["turns"]))
    right = sum(n for q, n in counts.items() if first_id(q) == conversation["source_id"])
    print(f"{case_id:18}{route:14}{kept:>8} of 3{clean:>10} of 3"
          f"{(f'{right} of 3' if conversation['source_id'] else '-'):>20}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

conversation      route          product kept  no new number  right policy first


topic_switch      as typed             3 of 3         3 of 3              3 of 3


topic_switch      rewrite              3 of 3         3 of 3              3 of 3


topic_switch      tool call            3 of 3         3 of 3              3 of 3


topic_switch      product rule         3 of 3         3 of 3              3 of 3


...

evidence_gap      product rule         3 of 3         3 of 3                   -


two_products      as typed             0 of 3         3 of 3                   -


two_products      rewrite              3 of 3         3 of 3                   -


two_products      tool call            0 of 3         3 of 3                   -


two_products      product rule         3 of 3         3 of 3                   -

gpt-4o-mini at https://api.openai.com/v1


### Count query instances without inventing additional evidence

For each distinct query `q` in a counter, `n` is its observed frequency. A passing check adds `n`, so three equal
passing queries contribute 3. `first_id(q)` performs one search per distinct query within that row, not one search
per counted instance. The displayed retrieval counts therefore weight a searched query by its frequency; they are
not three fresh retrieval runs when the query text repeated.

For the five answerable cases, there are `5 × 3 = 15` query instances per route. Product/digit columns include all
seven, `7 × 3 = 21`. Those denominators describe different questions. The final line names the chat model and
endpoint; the query embeddings use the embedding configuration printed in Config.


Read the outcome for each conversation before averaging:

| Case | As typed: right policy first | Rewrite | Original tool | Product rule |
|---|---|---|---|---|
| `topic_switch` | 3/3 | 3/3 | 3/3 | 3/3 |
| `link_after_boreal` | 0/3 | 3/3 | 0/3 | 3/3 |
| `account_type` | 3/3 | 3/3 | 3/3 | 3/3 |
| `pronoun` | 3/3 | 3/3 | 3/3 | 3/3 |
| `earlier_answer` | 3/3 | 3/3 | 3/3 | 3/3 |

Thus rewrite versus as typed gains three instances on **one** case and loses none under this top-ID check.
Original tool versus product rule changes the same case's top ID. The 15/15 rewrite result is five tested
conversations repeated three times, not fifteen independent conversations. No final answers are generated by this
table: `topic_switch` even has a passing top ID as typed despite Part A's refusals.

Product preservation tells a different story. As typed fails on five cases, although three of those still retrieve
the labelled policy first. The original tool drops products on `link_after_boreal`, `account_type` and `two_products`.
The rewrite and product rule retain the labelled products in all 21 instances, but naming both for `two_products`
does not resolve the ambiguity.

The rewrite fails the digit screen on 2 of 3 `earlier_answer` queries, because *H05* contributes `'05'`.
The remaining monthly-only query passes because monthly has no digits. Even a clean digit check would not establish that a query contains no copied answer.

Test a different boundary next. `evidence_gap` has these turns:

> user: How often are Atlas invoices sent?  
> assistant: Atlas invoices are sent monthly [H05].  
> user: What about Boreal?

The intended query is about **Boreal invoice frequency**. H05 describes Atlas only, so this corpus cannot supply
a Boreal frequency. The rewrite route below prints its query and candidates before generating three answers:


In [35]:
# gap: the evidence_gap conversation; the rewrite route: its query, its hits, three answers
gap = by_id["evidence_gap"]
gap_query = rewrite(gap["turns"])
print(gap_query)
gap_hits = lc_pipeline.retrieve(store, gap_query)
show(gap_hits)
answers(gap_query, gap_hits)

How often are Boreal invoices sent?


  0.5380  H05          Atlas invoices are sent monthly. Workspace owners can download invoice
  0.3801  H04          Boreal reset links expire after 10 minutes. Password reset uses a mobi
  0.2491  H03          Atlas exports reports as CSV files. Only workspace owners can export a


  3 of 3: Insufficient evidence. | cited, quote found: []


The rewrite is *How often are Boreal invoices sent?* H05, Atlas billing, ranks first at about 0.54, followed by
H04 and H03. The candidate preview identifies them; their texts concern Atlas billing, Boreal resets and Atlas
exports. None establishes a Boreal invoice frequency. The three cited-answer replies say *Insufficient evidence.*

Now run the product-rule tool loop on the **same original turns** three times. It receives history as well as tool
results and returns free text, so compare its actual meaning with the rewrite route rather than requiring the same
refusal string. This helper prints queries and final text; it does not expose intermediate ToolMessages as D did:


In [36]:
# The tool route with the product rule on evidence_gap, three times
for _ in range(3):
    print(tool_answer(gap["turns"], with_named_search))

(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only information about Atlas invoices being sent monthly is available [H05].')


(['Boreal invoices frequency'], 'The documents do not specify the frequency of Boreal invoices. Only the frequency for Atlas invoices is mentioned, which are sent monthly [H05].')


(['Boreal invoices frequency'], 'The documents do not specify how often Boreal invoices are sent. Only information about Atlas invoices being sent monthly is available [H05].')


All three saved tool replies say the documents lack Boreal billing information, while mentioning Atlas's monthly
invoices with H05. That is a qualified comparison, not an assertion that Boreal bills monthly. The rewrite route
refuses with its fixed string. Both avoid transferring Atlas's frequency to Boreal in this example.

Seven constructed cases provide diagnostic coverage of these particular failures. Repeated calls measure stability
on their fixed wording; they do not estimate how frequently real users encounter them. The product/digit screens
and nearest-ID comparisons are useful evidence to inspect, not an aggregate score of conversation quality.

### Checkpoint E

| Measure | Saved observation |
|---|---|
| Rewrite: required products present | 21/21 query instances across seven cases |
| Rewrite: answering policy first | 15/15 instances across five cases |
| `earlier_answer` citation copied into rewrite | 2/3; the digit screen flags `'05'` |
| Original tool's missing products | Three cases, three calls each |
| Product-rule query checks | All displayed applicable checks pass, including the unresolved two-product query |
| Actual `evidence_gap` replies | Rewrite refuses 3/3; tool states no Boreal billing information 3/3 |


---

## Part F: Ask for the product when history permits two readings (self-study)

A query can name both products and pass the screen while leaving the user's intent undecided.
`two_products` contains this complete exchange:

> user: How do I reset my Atlas password?  
> assistant: Atlas passwords can be reset by email [H01].  
> user: My colleague uses Boreal. How does it work there?  
> assistant: Boreal password reset uses a mobile app [H04].  
> user: How long is the link valid?

The case is deliberately labelled to require clarification: the speaker might mean their own Atlas link or the
colleague's Boreal link. Following the most recent product would choose one interpretation. Asking *Which product
do you mean, Atlas or Boreal?* lets the user settle it.

This differs from `evidence_gap`: there we know the intended product but lack its billing evidence. Clarifying the
product cannot create a missing policy. First try adding a clarification instruction to the rewrite prompt, once
per case, with `two_products` as the sole expected ask:


In [37]:
# ASK: one more line for the rewrite instruction; each conversation is rewritten once with it
ASK = (" If the conversation does not make clear which product the last turn is about,"
       " return only: Which product do you mean, Atlas or Boreal?")
for conversation in conversations:
    text = chat.invoke([("system", REWRITE + ASK), ("user", history_text(conversation["turns"]))]).content
    print(f"{conversation['id']:18}{text}")

topic_switch      Which product do you mean, Atlas or Boreal?


link_after_boreal How long is the password reset link valid for Boreal accounts?


account_type      Can temporary accounts reset their Atlas password by email?


pronoun           Which product do you mean, Atlas or Boreal?


earlier_answer    Which product do you mean, Atlas or Boreal?


evidence_gap      Which product do you mean, Atlas or Boreal?


two_products      Which product do you mean, Atlas or Boreal?


The saved prompt asks on five of seven cases: the ambiguous one and four clear cases, including `topic_switch`,
whose last turn explicitly names Boreal. Under these labels that is one desired ask and four unnecessary asks;
`link_after_boreal` and `account_type` are the two clear cases it rewrites. A count of five asks alone would hide
which decisions were right.

For these named-product fixtures, define a conservative rule: ask if user messages have mentioned more than one
product and the last user message names none. The code can apply that condition consistently. Choosing that
condition is a design decision; it is not a general solution to ambiguity:


In [38]:
# named() returns the products a text mentions; needs_question() applies the rule to the user's turns
PRODUCTS = ("Atlas", "Boreal")

def named(text):
    return {product for product in PRODUCTS if product.lower() in text.lower()}

def needs_question(turns):
    said = [turn["content"] for turn in turns if turn["role"] == "user"]
    return len(named(" ".join(said))) > 1 and not named(said[-1])

for conversation in conversations:
    print(f"{conversation['id']:18}{'ask which product' if needs_question(conversation['turns']) else 'rewrite and search'}")

topic_switch      rewrite and search
link_after_boreal rewrite and search
account_type      rewrite and search
pronoun           rewrite and search
earlier_answer    rewrite and search
evidence_gap      rewrite and search
two_products      ask which product


`named()` returns a **set**, so repeated mentions of Atlas still count as one product. `said` keeps only user
message strings, in order. Joining them finds all named products; `said[-1]` checks the latest user turn. The `and`
expression asks only when there is more than one distinct product overall and none in that last turn.

Only `two_products` triggers for these seven cases. Substring matching can miss aliases, and the function assumes
there is at least one user turn. It can also over-ask after a clear topic switch whose latest follow-up omits the
name. It cannot detect every ambiguous task or account type. A verified account context and an explicit product
selection can help; Lab 05's metadata filter showed how to restrict retrieval once that selection is known.

Join the demonstrated steps next. `answer_follow_up()` returns the clarification string without any model or
embedding call when the rule fires. Otherwise it rewrites, retrieves and answers. It does not enforce Part C's query
screens or validate the final claims; the saved output is a demonstration of this composition, not a release gate:


In [39]:
# answer_follow_up(): the question back when the rule says so, otherwise rewrite, search and answer
def answer_follow_up(turns):
    if needs_question(turns):
        return "Which product do you mean, Atlas or Boreal?"
    query = rewrite(turns)
    return lc_pipeline.answer(query, lc_pipeline.evidence(lc_pipeline.retrieve(store, query)), settings).answer

for case_id in ("link_after_boreal", "two_products"):
    print(f"{case_id:18}{answer_follow_up(by_id[case_id]['turns'])}")

link_after_boreal Boreal reset links expire after 10 minutes.
two_products      Which product do you mean, Atlas or Boreal?


The first case returns *Boreal reset links expire after 10 minutes.* The second returns a clarification question.
These two outputs test the two branches of the combined function; they do not establish correctness on unseen
conversations. The clarification branch is deterministic for these inputs and avoids an unnecessary search.

### Checkpoint F

| Decision | Saved observation |
|---|---|
| Added prompt line, seven cases once each | Five asks: one expected and four unnecessary under the case labels |
| Code rule on the same cases | Asks only for `two_products` |
| Combined function | Boreal's 10 minutes for `link_after_boreal`; clarification for `two_products` |


---

## What you can do now

- Trace the path from conversation to query, retrieved text and answer to diagnose a wrong-product reply.
- Rewrite a follow-up while keeping its product, task and conditions, then inspect what was dropped or copied.
- Apply product and digit screens with their limits, keeping them separate from evidence support.
- Inspect tool schemas and proposed arguments, execute a call, and return a matching `ToolMessage` before looping.
- Compare the same conversation across routes and report repeat counts with the number of distinct cases.
- Distinguish a missing policy from uncertain intent, and explain when this lab's clarification rule asks too much.


## Challenge: Compare meaning, query checks and final replies on a product switch

Task 1 applies the demonstrated routes to one new conversation; Task 2 is optional. Keep the actual queries,
answers and your explanations. The worked-solution observations below provide comparisons for your own run.

1. **Test a switch from Atlas email resets to Boreal.** The turns are:

   > user: Can I reset my Atlas password by email?  
   > assistant: Yes. Atlas passwords can be reset by email [H01].  
   > user: Does Boreal work that way too?

   Under Task 1, write a case dictionary with those turns in order, `products: ['Boreal']` and `source_id: 'H04'`.
   H04 says Boreal resets use a mobile app, so the intended comparison needs a no answer with that method.
   The supplied cell tests three answer routes: as typed, rewrite, and the product-rule tool loop. It does not run
   the original-description tool route from Part E.

   **Check:** read each route's queries and three replies. In the saved worked solution, as typed says *Yes, Boreal
   works that way too.* 3/3, citing H04 with quotes found. The rewrite *Can I reset my Boreal password by email?*
   passes both query screens but produces *Insufficient evidence.* 3/3. The tool route searches *Boreal password
   reset email* and gives no, with the mobile-app method, 3/3. These are example outcomes, not required wording.

   **Explain:** what does *that way* inherit from history? Why can an H04 quote occur in the evidence yet fail to
   support a yes to email resets? Do the product/digit screens judge that answer at all? Compare the rewrite's
   lost comparative wording with the tool route's access to history, and explain why this experiment does not
   isolate query wording as the sole cause of different replies.

2. **Optional: compare a less specific rewrite instruction.** Use this older tutorial instruction as the value of
   `OLD_REWRITE`: *Given a chat history and the latest user question which might reference context in the chat
   history, formulate a standalone question which can be understood without the chat history. Do NOT answer the
   question, just reformulate it if needed and otherwise return it as is.*

   The supplied cell makes three rewrite calls per original conversation. It tests the instruction as a string;
   no legacy retriever package is required.

   **Check:** inspect every failed query beside its case. The saved solution has 6/21 failing the product check:
   `link_after_boreal` and `two_products`, 3/3 each, produce *How long is the password reset link valid for?*
   No added digits are found. The two-product case still requires clarification even if a rewrite names both.

   **Explain:** which preservation instruction does `REWRITE` add? What information is absent from these six
   queries? Distinguish two failing conversations from six independent failures on new conversations, and say
   what task or account-type losses would remain invisible to these checks.


**Task 1.** Complete the conversation dictionary before running. Inspect its three turns and labels, then
compare the as-typed, rewritten and product-rule tool answers. Their common input is the conversation; their
answer prompts and output formats differ, as Part D explained.


In [40]:
# Task 1: your conversation as a dictionary, then every route on it
new_case = None    # TODO: {"id": ..., "slice": ..., "turns": [...], "products": ["Boreal"], "source_id": "H04"}

if new_case is None:
    print("⚠️ Write the conversation, then run the cell again.")
else:
    last = new_case["turns"][-1]["content"]
    print("as typed:"); answers(last, lc_pipeline.retrieve(store, last))
    new_query = rewrite(new_case["turns"])
    print(f"rewrite: {new_query!r} | product kept {products_kept(new_query, new_case)}"
          f" | new numbers {new_numbers(new_query, new_case['turns'])}")
    answers(new_query, lc_pipeline.retrieve(store, new_query))
    print("tool, product rule:")
    for _ in range(3):
        print(" ", tool_answer(new_case["turns"], with_named_search))

⚠️ Write the conversation, then run the cell again.


**Task 2 (optional).** Set `OLD_REWRITE` to the supplied instruction. Each printed line is one rewrite and its
product/digit checks; the cell makes 21 chat calls and generates no final answers. Retain the case ID beside each
query so a failing count can be traced to the lost information.


In [41]:
# Task 2: the older instruction, then three rewrites per conversation, each checked
OLD_REWRITE = None    # TODO: the instruction from the task, as a string

if OLD_REWRITE is None:
    print("⚠️ Set OLD_REWRITE, then run the cell again.")
else:
    for case in conversations:
        for _ in range(3):
            old = chat.invoke([("system", OLD_REWRITE), ("user", history_text(case["turns"]))]).content
            print(f"{case['id']:18}product kept {products_kept(old, case)!s:6}"
                  f"new numbers {new_numbers(old, case['turns'])!s:10}{old!r}")

⚠️ Set OLD_REWRITE, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: Function must have a docstring if description not provided.` | `@tool` on a function with no docstring | Add a docstring, or pass `description=` as Part D does for `search_named`; the model reads it |
| `IndexError: list index out of range` at `reply.tool_calls[0]` | the model answered in text and called no tool (a response may contain text without a tool proposal) | Read `reply.tool_calls` before you index it, as `tool_query()` and `tool_answer()` do |
| `OpenAIInvalidRequestError: Error code: 400 ... An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'.` | the model's tool call was sent back without a `ToolMessage` for each call | Append a result for every call before the next `invoke()`, as `tool_answer()` does |
| `OpenAIInvalidRequestError: Error code: 400 ... Function tools with reasoning_effort are not supported for gpt-6-luna in /v1/chat/completions.` | `REASONING_EFFORT` is `low`, `medium` or `high` for a reasoning model | Set `REASONING_EFFORT=none` in `.env`: Chat Completions takes tools on `gpt-6-luna` only at that effort |

## Resources

- [LangChain: retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval): retrieval pipelines,
  2-step RAG and retrieval through tools
- [LangChain: tools](https://docs.langchain.com/oss/python/langchain/tools): `@tool`, descriptions and argument
  schemas
- [LangChain: models, tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling): `bind_tools`,
  `tool_calls`, the execution loop and `ToolMessage`
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): what the API receives
  for a tool and returns for a call

Next: [Lab 10](../10_Long_Context_Or_Retrieval/README.md), *How to choose between long context and retrieval*. Back to [the lab list](../README.md).